# GAT Training on Colab (Converted from SLURM)

This notebook reproduces the training flow of the original SLURM script for running on Google Colab.

- Environment setup and GPU check
- Install dependencies (auto-match with current Torch/CUDA for torch-geometric)
- Obtain project code (Git clone or upload via Drive)
- Configure hyper-parameters
- Run training script `GAT/train_gat_graphsage_adv_nsfix.py`
- Save logs and models

Note: Code cells and comments are in English. Adjust paths if you use Google Drive instead of Git.


In [1]:
# Detect GPU and show environment info
import os, subprocess, sys, json

# Show GPU info if available
try:
    from google.colab import output  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print(f"Running in Colab: {IN_COLAB}")

!nvidia-smi || echo "No NVIDIA GPU detected"
!python -V
!pip -V

# Optional: set environment variables similar to SLURM script
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
print("Set PYTORCH_CUDA_ALLOC_CONF and CUDA_LAUNCH_BLOCKING.")


Running in Colab: True
Fri Aug  8 01:49:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.20             Driver Version: 570.133.20     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------

In [2]:
# Install dependencies (auto-match torch/torch_geometric)
import sys, subprocess, os

# Base Python packages
base_packages = [
    'packaging', 'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn'
]

# Install base packages
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--progress-bar', 'off', *base_packages])

# Detect torch; if not present, install a recent CUDA build suitable for Colab
try:
    import torch  # type: ignore
    print('Found torch', torch.__version__, 'CUDA available:', torch.cuda.is_available(), 'CUDA version:', getattr(torch.version, 'cuda', None))
except Exception:
    # Fallback: Install a recent torch (Colab typically ships torch with CUDA 12.1)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--progress-bar', 'off', 'torch', 'torchvision', 'torchaudio'])
    import torch  # type: ignore
    print('Installed torch', torch.__version__, 'CUDA available:', torch.cuda.is_available(), 'CUDA version:', getattr(torch.version, 'cuda', None))

# Build PyG wheel index URL based on torch and CUDA
from packaging import version

torch_version_str = torch.__version__.split('+')[0]
cuda_ver = getattr(torch.version, 'cuda', None)
if torch.cuda.is_available() and cuda_ver:
    cuda_tag = 'cu' + cuda_ver.replace('.', '')
else:
    cuda_tag = 'cpu'
index_url = f"https://data.pyg.org/whl/torch-{torch_version_str}+{cuda_tag}.html"
print('PyG wheels index:', index_url)

# Install torch-geometric and extensions
pyg_packages = [
    'pyg-lib', 'torch-scatter', 'torch-sparse', 'torch-cluster', 'torch-spline-conv', 'torch-geometric'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--progress-bar', 'off', '-f', index_url, *pyg_packages])

import torch_geometric  # type: ignore
print('Installed torch_geometric', torch_geometric.__version__)


Found torch 2.6.0+cu124 CUDA available: True CUDA version: 12.4
PyG wheels index: https://data.pyg.org/whl/torch-2.6.0+cu124.html
Installed torch_geometric 2.6.1


In [3]:
# Overwrite adv script to natively include LaTeX logging (no patch needed)
from pathlib import Path

adv_code_with_table = r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
GAT Training Script based on GraphSage.ipynb (with LaTeX logging)
"""
import os
import sys
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import json
import gc
from datetime import datetime
from collections import defaultdict
import random
import argparse
from typing import Optional

# PyTorch Geometric imports
import torch_geometric
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import negative_sampling, to_undirected
from torch_geometric.loader import LinkNeighborLoader

# Scikit-learn imports
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler

class FocalLoss(torch.nn.Module):
    def __init__(self, gamma: float = 2.0, pos_weight: Optional[torch.Tensor] = None):
        super().__init__()
        self.gamma = gamma
        self.register_buffer("pos_weight", pos_weight if pos_weight is not None else torch.tensor(1.0))
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none", pos_weight=self.pos_weight)
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_factor = (1 - p_t) ** self.gamma
        loss = focal_factor * bce_loss
        return loss.mean()

class MixedLoss(torch.nn.Module):
    def __init__(self, alpha: float = 0.5, focal_gamma: float = 1.5, pos_weight: Optional[torch.Tensor] = None):
        super().__init__()
        self.alpha = alpha
        self.bce = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight) if pos_weight is not None else torch.nn.BCEWithLogitsLoss()
        self.focal = FocalLoss(gamma=focal_gamma, pos_weight=pos_weight)
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return self.alpha * self.bce(logits, targets) + (1 - self.alpha) * self.focal(logits, targets)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def setup_logging():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_dir = Path("logs")
    log_dir.mkdir(exist_ok=True)
    log_file = log_dir / f"gat_training_{timestamp}.log"
    class Logger:
        def __init__(self, filename):
            self.terminal = sys.stdout
            self.log = open(filename, 'w', encoding='utf-8')
        def write(self, message):
            self.terminal.write(message)
            self.log.write(message)
            self.log.flush()
        def flush(self):
            self.terminal.flush()
            self.log.flush()
    sys.stdout = Logger(log_file)
    sys.stderr = Logger(log_file)
    return log_file

def log_message(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {message}")

class DataProcessor:
    def __init__(self, csv_path):
        self.csv_path = csv_path
        self.df = None
        self.asin2idx = None
        self.feature_cols = None
        self.scaler = StandardScaler()
    def load_and_process(self):
        log_message(f"Loading data from {self.csv_path}...")
        self.df = pd.read_csv(self.csv_path)
        self.df.columns = self.df.columns.str.strip()
        log_message(f"Loaded {len(self.df)} items")
        self.asin2idx = {asin: i for i, asin in enumerate(self.df['ASIN'])}
        log_message(f"Created mapping for {len(self.asin2idx)} unique ASINs")
        self._process_features()
        edge_index = self._build_graph()
        x = self._create_node_features()
        data = Data(x=x, edge_index=edge_index)
        del self.df
        gc.collect()
        return data
    def _process_features(self):
        log_message("Processing features...")
        self.feature_cols = [
            'salesrank_log','reviews_total_log','reviews_downloaded_log',
            'reviews_avg_ratings','reviews_avg_votes','reviews_avg_helpful','category_count']
        self.df[self.feature_cols] = self.df[self.feature_cols].fillna(0.0)
        for col in ['salesrank_log','reviews_total_log','reviews_downloaded_log']:
            if col in self.df.columns:
                self.df[col] = np.log1p(self.df[col].clip(lower=0))
        features = self.df[self.feature_cols].values
        self.df[self.feature_cols] = self.scaler.fit_transform(features)
        log_message(f"Processed {len(self.feature_cols)} features")
    def _build_graph(self):
        log_message("Building graph from similarity relationships...")
        src, dst = [], []
        for idx, row in self.df.iterrows():
            sims = str(row['similar']).split(',') if pd.notna(row['similar']) else []
            for s in sims:
                s = s.strip()
                if s and s in self.asin2idx:
                    j = self.asin2idx[s]
                    if idx != j:
                        src.append(idx); dst.append(j)
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        edge_index = to_undirected(edge_index)
        log_message(f"Graph built: {len(self.asin2idx)} nodes, {edge_index.size(1)//2} edges")
        return edge_index
    def _create_node_features(self):
        return torch.tensor(self.df[self.feature_cols].values, dtype=torch.float)

class GATModel(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=8, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.convs = torch.nn.ModuleList()
        self.convs.append(GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout, concat=True))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=dropout, concat=True))
        if num_layers > 1:
            expected_out_dim = hidden_dim * heads
            if out_dim != expected_out_dim:
                print(f"Warning: Model out_dim ({out_dim}) != expected ({expected_out_dim}), using {expected_out_dim}")
                out_dim = expected_out_dim
            self.convs.append(GATConv(hidden_dim * heads, out_dim, heads=1, dropout=dropout, concat=False))
        else:
            if out_dim > hidden_dim:
                print(f"Warning: Single layer out_dim ({out_dim}) > hidden_dim ({hidden_dim}), using {hidden_dim}")
                out_dim = hidden_dim
            self.convs.append(GATConv(in_dim, out_dim, heads=1, dropout=dropout, concat=False))
    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.elu(x)
        return x

def dot_product_decode(z, edge_index):
    return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=1)

class GATTrainer:
    def __init__(self, model, device, lr: float = 0.01, weight_decay: float = 5e-4, *, criterion: Optional[torch.nn.Module] = None):
        self.model = model
        self.device = device
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = criterion if criterion is not None else torch.nn.BCEWithLogitsLoss()
        self.train_losses, self.val_losses = [], []
        self.train_aucs, self.val_aucs = [], []
    def train_epoch(self, train_data, neg_ratio=1):
        self.model.train()
        neg_edge_index = negative_sampling(edge_index=train_data.edge_index, num_nodes=train_data.num_nodes, num_neg_samples=int(train_data.edge_index.size(1) * neg_ratio))
        edge_label_index = torch.cat([train_data.edge_index, neg_edge_index], dim=1)
        edge_label = torch.cat([torch.ones(train_data.edge_index.size(1)), torch.zeros(neg_edge_index.size(1))], dim=0).to(self.device)
        self.optimizer.zero_grad()
        z = self.model(train_data.x.to(self.device), train_data.edge_index.to(self.device))
        out = dot_product_decode(z, edge_label_index.to(self.device))
        loss = self.criterion(out, edge_label)
        loss.backward(); self.optimizer.step()
        return loss.item()
    def evaluate(self, data, neg_ratio=1, split_name="val"):
        self.model.eval()
        with torch.no_grad():
            neg_edge_index = negative_sampling(edge_index=data.edge_index, num_nodes=data.num_nodes, num_neg_samples=int(data.edge_index.size(1) * neg_ratio))
            edge_label_index = torch.cat([data.edge_index, neg_edge_index], dim=1)
            edge_label = torch.cat([torch.ones(data.edge_index.size(1)), torch.zeros(neg_edge_index.size(1))], dim=0).to(self.device)
            z = self.model(data.x.to(self.device), data.edge_index.to(self.device))
            out = dot_product_decode(z, edge_label_index.to(self.device))
            loss = self.criterion(out, edge_label)
            auc = roc_auc_score(edge_label.cpu().numpy(), out.cpu().numpy())
            ap = average_precision_score(edge_label.cpu().numpy(), out.cpu().numpy())
            return loss.item(), auc, ap
    def plot_training_history(self, save_path):
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes[0, 0].plot(self.train_losses, label='Train Loss'); axes[0, 0].plot(self.val_losses, label='Val Loss'); axes[0, 0].legend(); axes[0, 0].grid(True)
        axes[0, 1].plot(self.train_aucs, label='Train AUC'); axes[0, 1].plot(self.val_aucs, label='Val AUC'); axes[0, 1].legend(); axes[0, 1].grid(True)
        axes[1, 0].hist(self.train_losses, bins=20, alpha=0.7, label='Train Loss'); axes[1, 0].hist(self.val_losses, bins=20, alpha=0.7, label='Val Loss'); axes[1, 0].legend()
        axes[1, 1].hist(self.train_aucs, bins=20, alpha=0.7, label='Train AUC'); axes[1, 1].hist(self.val_aucs, bins=20, alpha=0.7, label='Val AUC'); axes[1, 1].legend()
        plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches='tight'); plt.close(); log_message(f"Training history plot saved to {save_path}")

def validate_args(args):
    if args.num_layers == 1:
        if args.heads != 1:
            print(f"Warning: Single layer detected, auto-fixing heads from {args.heads} to 1"); args.heads = 1
        if args.out_dim > args.hidden_dim:
            print(f"Warning: Single layer out_dim ({args.out_dim}) > hidden_dim ({args.hidden_dim}), auto-fixing out_dim to {args.hidden_dim}"); args.out_dim = args.hidden_dim
    elif args.num_layers > 1:
        expected_out_dim = args.hidden_dim * args.heads
        if args.out_dim != expected_out_dim:
            print(f"Warning: out_dim ({args.out_dim}) != hidden_dim*heads ({expected_out_dim})"); print(f"Auto-fixing out_dim from {args.out_dim} to {expected_out_dim}"); args.out_dim = expected_out_dim
    return args

def get_args():
    p = argparse.ArgumentParser(description='GAT Training based on GraphSage.ipynb')
    p.add_argument('--epochs', type=int, default=100)
    p.add_argument('--lr', type=float, default=0.01)
    p.add_argument('--weight_decay', type=float, default=5e-4)
    p.add_argument('--patience', type=int, default=20)
    p.add_argument('--hidden_dim', type=int, default=32)
    p.add_argument('--out_dim', type=int, default=16)
    p.add_argument('--heads', type=int, default=2)
    p.add_argument('--num_layers', type=int, default=2)
    p.add_argument('--dropout', type=float, default=0.2)
    p.add_argument('--neg_ratio', type=float, default=1.0)
    p.add_argument('--inductive', action='store_true')
    p.add_argument('--pos_weight', type=float, default=1.0)
    p.add_argument('--use_focal', action='store_true')
    p.add_argument('--focal_gamma', type=float, default=2.0)
    p.add_argument('--mix_alpha', type=float, default=0.5)
    p.add_argument('--scheduler', type=str, choices=['none','multistep','cosine'], default='none')
    p.add_argument('--milestones', type=str, default='80,120')
    p.add_argument('--lr_gamma', type=float, default=0.1)
    return validate_args(p.parse_args())

def main():
    args = get_args()
    set_seed(42)
    log_file = setup_logging()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.set_per_process_memory_fraction(0.8)
    log_message("Starting GAT training based on GraphSage.ipynb pipeline")
    log_message(f"Log file: {log_file}")
    log_message(f"Arguments: {args}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    log_message(f"Using device: {device}")
    data_processor = DataProcessor('data/items_cleaned.csv')
    data = data_processor.load_and_process()
    # Split (inductive vs transductive)
    if args.inductive:
        num_nodes = data.num_nodes
        all_nodes = torch.arange(num_nodes)
        num_train_nodes = int(0.7 * num_nodes)
        num_val_nodes = int(0.15 * num_nodes)
        node_indices = torch.randperm(num_nodes)
        train_nodes = all_nodes[node_indices[:num_train_nodes]]
        val_nodes = all_nodes[node_indices[num_train_nodes:num_train_nodes+num_val_nodes]]
        test_nodes = all_nodes[node_indices[num_train_nodes+num_val_nodes:]]
        train_mask = torch.zeros(num_nodes, dtype=torch.bool); train_mask[train_nodes]=True
        val_mask = torch.zeros(num_nodes, dtype=torch.bool); val_mask[val_nodes]=True
        test_mask = torch.zeros(num_nodes, dtype=torch.bool); test_mask[test_nodes]=True
        def split_edges_by_nodes(edge_index):
            train_edges, val_edges, test_edges = [], [], []
            for i in range(edge_index.size(1)):
                src, dst = edge_index[0, i], edge_index[1, i]
                if train_mask[src] and train_mask[dst]: train_edges.append(i)
                elif val_mask[src] and val_mask[dst]: val_edges.append(i)
                elif test_mask[src] and test_mask[dst]: test_edges.append(i)
            return train_edges, val_edges, test_edges
        train_edge_indices, val_edge_indices, test_edge_indices = split_edges_by_nodes(data.edge_index)
        train_data = Data(x=data.x, edge_index=data.edge_index[:, train_edge_indices], num_nodes=data.num_nodes)
        val_data   = Data(x=data.x, edge_index=data.edge_index[:, val_edge_indices],   num_nodes=data.num_nodes)
        test_data  = Data(x=data.x, edge_index=data.edge_index[:, test_edge_indices],  num_nodes=data.num_nodes)
        train_data.train_mask, train_data.val_mask, train_data.test_mask = train_mask, val_mask, test_mask
    else:
        transform = RandomLinkSplit(num_val=0.1, num_test=0.1, is_undirected=True, add_negative_train_samples=False)
        train_data, val_data, test_data = transform(data)
        num_nodes = data.num_nodes
        train_data.train_mask = torch.ones(num_nodes, dtype=torch.bool)
        train_data.val_mask = torch.ones(num_nodes, dtype=torch.bool)
        train_data.test_mask = torch.ones(num_nodes, dtype=torch.bool)
    in_dim = data.x.size(1)
    model = GATModel(in_dim=in_dim, hidden_dim=args.hidden_dim, out_dim=args.out_dim, heads=args.heads, num_layers=args.num_layers, dropout=args.dropout).to(device)
    # Loss selection
    pos_weight_tensor = torch.tensor(args.pos_weight, device=device) if args.pos_weight != 1.0 else None
    if args.use_focal:
        criterion = FocalLoss(gamma=args.focal_gamma, pos_weight=pos_weight_tensor)
    elif args.mix_alpha is not None and 0.0 < args.mix_alpha < 1.0:
        criterion = MixedLoss(alpha=args.mix_alpha, focal_gamma=args.focal_gamma, pos_weight=pos_weight_tensor)
    else:
        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor) if pos_weight_tensor is not None else torch.nn.BCEWithLogitsLoss()
    trainer = GATTrainer(model, device, lr=args.lr, weight_decay=args.weight_decay, criterion=criterion)
    # Scheduler
    scheduler = None
    if args.scheduler == 'multistep':
        milestones = [int(m.strip()) for m in args.milestones.split(',') if m.strip()]
        scheduler = torch.optim.lr_scheduler.MultiStepLR(trainer.optimizer, milestones=milestones, gamma=args.lr_gamma)
    elif args.scheduler == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(trainer.optimizer, T_max=args.epochs, eta_min=args.lr * 0.01)
    # LaTeX table header
    _gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    _model_name = f"GAT(h{args.hidden_dim},hd{args.heads},L{args.num_layers})"
    _table_header_1 = r"\\textbf{Model} & \\textbf{GPU} & \\multicolumn{3}{c}{\\textbf{Training}} & \\multicolumn{3}{c}{\\textbf{Validation}} & \\textbf{Epoch Time (s)} \\\\"
    _table_header_2 = r"  &  & AUC & AP & Loss & AUC & AP & Loss & \\\\"
    _timestamp_tbl = Path(str(log_file)).stem.replace('gat_training_', '')
    _table_path = Path('logs') / f"gat_training_table_{_timestamp_tbl}.tex"
    with open(_table_path, 'w', encoding='utf-8') as f:
        f.write(_table_header_1 + "\n" + _table_header_2 + "\n")
    print(_table_header_1); print(_table_header_2)
    _table_rows, _epoch_times, _train_aps_hist, _val_aps_hist = [], [], [], []
    # Train loop
    best_val_auc = 0; patience_counter = 0
    log_message(f"Starting training for {args.epochs} epochs...")
    for epoch in range(args.epochs):
        start_time = time.time()
        train_loss = trainer.train_epoch(train_data, neg_ratio=args.neg_ratio)
        val_loss, val_auc, val_ap = trainer.evaluate(val_data, neg_ratio=args.neg_ratio, split_name="val")
        train_loss_eval, train_auc, train_ap = trainer.evaluate(train_data, neg_ratio=args.neg_ratio, split_name="train")
        trainer.train_losses.append(train_loss); trainer.val_losses.append(val_loss)
        trainer.train_aucs.append(train_auc);   trainer.val_aucs.append(val_auc)
        epoch_time = time.time() - start_time
        # Row output
        _row = f"{_model_name} & {_gpu_name} & {train_auc:.4f} & {train_ap:.4f} & {train_loss:.4f} & {val_auc:.4f} & {val_ap:.4f} & {val_loss:.4f} & {epoch_time:.2f} \\\\"
        print(_row)
        with open(_table_path, 'a', encoding='utf-8') as f:
            f.write(_row + "\n")
        _table_rows.append(_row); _epoch_times.append(epoch_time); _train_aps_hist.append(train_ap); _val_aps_hist.append(val_ap)
        log_message(f"Epoch {epoch+1:3d}/{args.epochs} - Train Loss: {train_loss:.4f}, Train AUC: {train_auc:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}, Time: {epoch_time:.2f}s")
        if scheduler is not None: scheduler.step()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        if val_auc > best_val_auc:
            best_val_auc = val_auc; patience_counter = 0
            model_dir = Path("models"); model_dir.mkdir(exist_ok=True)
            model_suffix = f"h{args.hidden_dim}_hd{args.heads}_neg{args.neg_ratio}"
            best_model_path = model_dir / f"gat_best_{model_suffix}.pt"
            torch.save(model.state_dict(), best_model_path)
            log_message(f"New best model saved with Val AUC: {val_auc:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= args.patience:
                log_message(f"Early stopping triggered after {epoch+1} epochs"); break
    # Test
    log_message("Training completed. Evaluating on test set...")
    model_suffix = f"h{args.hidden_dim}_hd{args.heads}_neg{args.neg_ratio}"
    best_model_path = f"models/gat_best_{model_suffix}.pt"
    model.load_state_dict(torch.load(best_model_path))
    test_loss, test_auc, test_ap = trainer.evaluate(test_data, neg_ratio=args.neg_ratio, split_name="test")
    log_message(f"Final Test Results - Loss: {test_loss:.4f}, AUC: {test_auc:.4f}, AP: {test_ap:.4f}")
    # Plot & JSON
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    plot_path = f"logs/gat_training_history_{timestamp}.png"; trainer.plot_training_history(plot_path)
    results = {
        'best_val_auc': best_val_auc,
        'test_loss': test_loss,
        'test_auc': test_auc,
        'test_ap': test_ap,
        'epochs_trained': len(trainer.train_losses),
        'model_config': {'in_dim': in_dim, 'hidden_dim': args.hidden_dim, 'out_dim': args.out_dim, 'heads': args.heads, 'num_layers': args.num_layers},
        'training_args': vars(args)
    }
    # Extra annotations
    results['gpu'] = _gpu_name
    results['model_name'] = _model_name
    results['table_file'] = str(_table_path)
    results['table_header'] = _table_header_1 + "\n" + _table_header_2
    results['table_rows'] = _table_rows
    results['epoch_times_s'] = _epoch_times
    results['train_aps'] = _train_aps_hist
    results['val_aps'] = _val_aps_hist
    results_path = f"logs/gat_training_results_{timestamp}.json"
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    log_message(f"Training results saved to {results_path}")
    log_message("Training completed successfully!")

if __name__ == "__main__":
    main()
'''

Path('GAT/train_gat_graphsage_adv.py').write_text(adv_code_with_table, encoding='utf-8')
print('Overwritten GAT/train_gat_graphsage_adv.py with native LaTeX logging')


Overwritten GAT/train_gat_graphsage_adv.py with native LaTeX logging


## Data setup

This project expects `data/items_cleaned.csv`.

- If using Google Drive: toggle `USE_DRIVE = True` below and set `DRIVE_DATA_DIR` to your folder containing `data/items_cleaned.csv`.
- Otherwise, upload `items_cleaned.csv` to a `data/` folder in the Colab workspace.


In [4]:
USE_DRIVE = False
DRIVE_DATA_DIR = '/content/drive/MyDrive/Group-Project'  # change if needed

from pathlib import Path

if USE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    !mkdir -p data
    # Copy only needed file to local runtime for faster IO
    !cp -v "$DRIVE_DATA_DIR/data/items_cleaned.csv" data/
else:
    Path('data').mkdir(exist_ok=True)

# Verify data exists
csv_path = Path('data/items_cleaned.csv')
print('Data exists:', csv_path.exists(), '->', csv_path)
assert csv_path.exists(), 'Please place data/items_cleaned.csv before running training.'


Data exists: True -> data/items_cleaned.csv


In [5]:
# # Hyper-parameters
EPOCHS        = 150
HIDDEN_DIM    = 256
HEADS         = 2
NUM_LAYERS    = 2
LR            = 0.005
WEIGHT_DECAY  = 1e-4
DROPOUT       = 0.05
NEG_RATIO     = 0.3
POS_WEIGHT    = 2.5
MIX_ALPHA     = 0.1
USE_FOCAL     = True
FOCAL_GAMMA   = 2.0
USE_INDUCTIVE = True
SCHEDULER     = 'multistep'
MILESTONES    = [60, 100, 140]


print({
    'epochs':       EPOCHS,
    'hidden_dim':   HIDDEN_DIM,
    'heads':        HEADS,
    'num_layers':   NUM_LAYERS,
    'lr':           LR,
    'weight_decay': WEIGHT_DECAY,
    'dropout':      DROPOUT,
    'neg_ratio':    NEG_RATIO,
    'pos_weight':   POS_WEIGHT,
    'mix_alpha':    MIX_ALPHA,
    'use_focal':    USE_FOCAL,
    'focal_gamma':  FOCAL_GAMMA,
    'inductive':    USE_INDUCTIVE,
    'scheduler':    SCHEDULER,
    'milestones':   MILESTONES,
})


{'epochs': 150, 'hidden_dim': 256, 'heads': 2, 'num_layers': 2, 'lr': 0.005, 'weight_decay': 0.0001, 'dropout': 0.05, 'neg_ratio': 0.3, 'pos_weight': 2.5, 'mix_alpha': 0.1, 'use_focal': True, 'focal_gamma': 2.0, 'inductive': True, 'scheduler': 'multistep', 'milestones': [60, 100, 140]}


In [6]:
# Run training (nsfix version)
import sys, shlex, subprocess

cmd = [
    sys.executable, '-u', 'GAT/train_gat_graphsage_adv_nsfix.py',
    '--epochs', str(EPOCHS),
    '--hidden_dim', str(HIDDEN_DIM),
    '--heads', str(HEADS),
    '--num_layers', str(NUM_LAYERS),
    '--lr', str(LR),
    '--weight_decay', str(WEIGHT_DECAY),
    '--dropout', str(DROPOUT),
    '--neg_ratio', str(NEG_RATIO),
    '--pos_weight', str(POS_WEIGHT),
    '--mix_alpha', str(MIX_ALPHA),
    '--scheduler', str(SCHEDULER),
]

if USE_INDUCTIVE:
    cmd.append('--inductive')

print('Running command:', ' '.join(shlex.quote(c) for c in cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('Exit code:', proc.returncode)
assert proc.returncode == 0, 'Training script failed'


Running command: /usr/bin/python3 -u GAT/train_gat_graphsage_adv_nsfix.py --epochs 150 --hidden_dim 256 --heads 2 --num_layers 2 --lr 0.005 --weight_decay 0.0001 --dropout 0.05 --neg_ratio 0.3 --pos_weight 2.5 --mix_alpha 0.1 --scheduler multistep --inductive
Auto-fixing out_dim from 16 to 512
[2025-08-08 01:51:18] Starting GAT training based on GraphSage.ipynb pipeline
[2025-08-08 01:51:18] Log file: logs/gat_training_20250808_015117.log
[2025-08-08 01:51:18] Arguments: Namespace(epochs=150, lr=0.005, weight_decay=0.0001, patience=20, hidden_dim=256, out_dim=512, heads=2, num_layers=2, dropout=0.05, neg_ratio=0.3, inductive=True, pos_weight=2.5, use_focal=False, focal_gamma=2.0, mix_alpha=0.1, scheduler='multistep', milestones='80,120', lr_gamma=0.1)
[2025-08-08 01:51:18] Using device: cuda
[2025-08-08 01:51:18] Loading data from data/items_cleaned.csv...
[2025-08-08 01:51:19] Loaded 402691 items
[2025-08-08 01:51:19] Created mapping for 402691 unique ASINs
[2025-08-08 01:51:19] Proce